# 🍔☕🍗 Starbucks vs KFC vs McDonald's — Sales Analysis & AI Model

---

| Field | Details |
|---|---|
| **Student Name** | M Abu Bakar |
| **SAP ID** | 80429 |
| **Subject** | Design and Analysis of Algorithms |
| **Instructor** | Sir Karar Haider |
| **Dataset Source** | Curated Dataset (2015–2024) |
| **Environment** | Google Colab |
| **Date** | 2026 |

---

## 📌 About the Dataset
Yearly (2015–2024) worldwide store count, employees, company revenue, and systemwide sales for **McDonald's**, **KFC**, and **Starbucks** — three of the largest global food & beverage chains. The goal is to compare which brand generates the most sales, and to train an **AI model** that can (a) predict a company's systemwide sales from its scale, and (b) classify which company a given year's stats belong to.

> ⚠️ Figures are approximate/illustrative (compiled from public annual-report ranges) for educational analysis — not official investor data.

## 📦 Step 1 — Install & Import Libraries

In [ ]:
!pip install pandas matplotlib seaborn scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, confusion_matrix, classification_report

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='darkgrid', palette='muted')

print('=' * 60)
print('  Student  : M Abu Bakar')
print('  SAP ID   : 80429')
print('  Subject  : Design and Analysis of Algorithms')
print('  Sir      : Karar Haider')
print('=' * 60)
print('✅ All libraries imported!')

## 📥 Step 2 — Load the Dataset

In [ ]:
import io

raw_data = """Company,Year,Stores_Worldwide,Employees,Revenue_Million_USD,Systemwide_Sales_Million_USD,Country_HQ,Founded
McDonald's,2015,36525,420000,25413,88500,USA,1955
McDonald's,2016,36899,235000,24622,92200,USA,1955
McDonald's,2017,37241,235000,22820,96500,USA,1955
McDonald's,2018,37855,210000,21025,101400,USA,1955
McDonald's,2019,38695,205000,21076,108200,USA,1955
McDonald's,2020,39198,200000,19208,93900,USA,1955
McDonald's,2021,40031,200000,23223,112500,USA,1955
McDonald's,2022,40275,150000,23183,118700,USA,1955
McDonald's,2023,41822,150000,25494,130400,USA,1955
McDonald's,2024,43000,150000,25920,135000,USA,1955
KFC,2015,20226,23000,1050,23600,USA,1952
KFC,2016,20574,23500,1100,24800,USA,1952
KFC,2017,22629,24000,1150,26800,USA,1952
KFC,2018,24104,24500,1200,28100,USA,1952
KFC,2019,24945,25000,1300,29600,USA,1952
KFC,2020,25823,24000,1200,27000,USA,1952
KFC,2021,26934,24500,1400,30200,USA,1952
KFC,2022,27581,25000,1500,32200,USA,1952
KFC,2023,29388,25500,1600,34600,USA,1952
KFC,2024,30000,26000,1700,36000,USA,1952
Starbucks,2015,23768,238000,19163,19163,USA,1971
Starbucks,2016,25085,254000,21316,21316,USA,1971
Starbucks,2017,27339,277000,22387,22387,USA,1971
Starbucks,2018,29324,291000,24720,24720,USA,1971
Starbucks,2019,31256,346000,26509,26509,USA,1971
Starbucks,2020,32660,349000,23518,23518,USA,1971
Starbucks,2021,33833,383000,29061,29061,USA,1971
Starbucks,2022,35711,402000,32250,32250,USA,1971
Starbucks,2023,38038,381000,35976,35976,USA,1971
Starbucks,2024,40199,385000,36180,36180,USA,1971"""

df = pd.read_csv(io.StringIO(raw_data))
print(f'✅ Dataset loaded!  Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')
df.head(10)

## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
print('=' * 65)
print('       FAST FOOD / COFFEE CHAIN DATASET — OVERVIEW')
print('       Student: M Abu Bakar  |  SAP ID: 80429')
print('       Subject: Design and Analysis of Algorithms')
print('       Instructor: Sir Karar Haider')
print('=' * 65)
print(f'  Total Rows          : {df.shape[0]}')
print(f'  Companies            : {", ".join(df.Company.unique())}')
print(f'  Year Range           : {df.Year.min()} — {df.Year.max()}')
latest = df[df.Year == df.Year.max()]
top_sales = latest.loc[latest.Systemwide_Sales_Million_USD.idxmax()]
print(f'  Highest 2024 Sales   : {top_sales.Company}  (${top_sales.Systemwide_Sales_Million_USD:,.0f}M)')
top_stores = latest.loc[latest.Stores_Worldwide.idxmax()]
print(f'  Most Stores (2024)   : {top_stores.Company}  ({top_stores.Stores_Worldwide:,} stores)')
print('=' * 65)

In [ ]:
print('📊 Statistical Summary by Company:')
df.groupby('Company')[['Stores_Worldwide','Employees','Revenue_Million_USD','Systemwide_Sales_Million_USD']].mean().round(0)

## 📊 Step 4 — Data Visualizations

In [ ]:
def add_stamp(ax):
    """Add student info watermark to every chart."""
    ax.text(0.99, 0.01, 'M Abu Bakar | SAP: 80429 | DAA | Sir Karar Haider',
            transform=ax.transAxes, fontsize=7.5, color='#555555',
            ha='right', va='bottom', alpha=0.75,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.5))

colors = {"McDonald's": '#DA291C', 'KFC': '#800000', 'Starbucks': '#00704A'}
print('✅ add_stamp() helper defined — every chart will carry your name.')

In [ ]:
# ── Chart 1 : Systemwide Sales Trend (2015-2024) ─────────────────
fig, ax = plt.subplots(figsize=(13, 6))
for company, grp in df.groupby('Company'):
    ax.plot(grp['Year'], grp['Systemwide_Sales_Million_USD'] / 1000, marker='o',
            label=company, color=colors[company], linewidth=2.4, markersize=6)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Systemwide Sales (Billion USD)', fontsize=12)
ax.set_title("💰  Systemwide Sales Trend (2015–2024)\nM Abu Bakar | SAP: 80429 | Design and Analysis of Algorithms | Sir Karar Haider",
             fontsize=12, fontweight='bold')
ax.legend(); add_stamp(ax); plt.tight_layout()
plt.savefig('chart1_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 1 saved.')

In [ ]:
# ── Chart 2 : 2024 Sales Comparison (who is winning) ─────────────
latest = df[df.Year == 2024].sort_values('Systemwide_Sales_Million_USD', ascending=False)
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(latest['Company'], latest['Systemwide_Sales_Million_USD'] / 1000,
              color=[colors[c] for c in latest['Company']], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, latest['Systemwide_Sales_Million_USD']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'${val/1000:.1f}B', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Systemwide Sales (Billion USD)', fontsize=12)
ax.set_title('🏆  2024 Sales Comparison — Who Is Winning?\nM Abu Bakar | SAP: 80429 | Design and Analysis of Algorithms | Sir Karar Haider',
             fontsize=12, fontweight='bold')
add_stamp(ax); plt.tight_layout()
plt.savefig('chart2_2024_sales_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Chart 2 saved.  Winner: {latest.iloc[0]['Company']}")

In [ ]:
# ── Chart 3 : Store Count Growth ─────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
for company, grp in df.groupby('Company'):
    ax.plot(grp['Year'], grp['Stores_Worldwide'], marker='s',
            label=company, color=colors[company], linewidth=2.4, markersize=6)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Stores Worldwide', fontsize=12)
ax.set_title('🏬  Store Count Growth (2015–2024)\nM Abu Bakar | SAP: 80429 | Design and Analysis of Algorithms | Sir Karar Haider',
             fontsize=12, fontweight='bold')
ax.legend(); add_stamp(ax); plt.tight_layout()
plt.savefig('chart3_store_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 3 saved.')

In [ ]:
# ── Chart 4 : Revenue vs Employees ───────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
for company, grp in df.groupby('Company'):
    ax.scatter(grp['Employees'] / 1000, grp['Revenue_Million_USD'] / 1000,
               label=company, color=colors[company], s=90, edgecolors='white', linewidth=0.8)
ax.set_xlabel('Employees (Thousands)', fontsize=12)
ax.set_ylabel('Revenue (Billion USD)', fontsize=12)
ax.set_title('👥  Revenue vs Employees\nM Abu Bakar | SAP: 80429 | Design and Analysis of Algorithms | Sir Karar Haider',
             fontsize=12, fontweight='bold')
ax.legend(); add_stamp(ax); plt.tight_layout()
plt.savefig('chart4_revenue_vs_employees.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 4 saved.')

In [ ]:
# ── Chart 5 : Correlation Heatmap ────────────────────────────────
num = df[['Year','Stores_Worldwide','Employees','Revenue_Million_USD','Systemwide_Sales_Million_USD']]
corr = num.corr()
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('🔗  Correlation Matrix\nM Abu Bakar | SAP: 80429 | Design and Analysis of Algorithms | Sir Karar Haider',
             fontsize=11, fontweight='bold')
add_stamp(ax); plt.tight_layout()
plt.savefig('chart5_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 5 saved.')

## 🤖 Step 5 — Train AI Model #1: Predict Systemwide Sales (Regression)

We train a **Linear Regression** model that predicts a company's `Systemwide_Sales_Million_USD` from `Year`, `Stores_Worldwide`, and `Employees`.

In [ ]:
X = df[['Year', 'Stores_Worldwide', 'Employees']]
y = df['Systemwide_Sales_Million_USD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
y_pred = reg_model.predict(X_test)

print('=' * 60)
print('  AI MODEL #1 — Sales Prediction (Linear Regression)')
print('=' * 60)
print(f'  R² Score            : {r2_score(y_test, y_pred):.3f}')
print(f'  Mean Absolute Error : ${mean_absolute_error(y_test, y_pred):,.1f}M')
print('=' * 60)

comparison = pd.DataFrame({'Actual_Sales_M': y_test.values, 'Predicted_Sales_M': y_pred.round(0)})
comparison

In [ ]:
# Predict 2025 sales for each company assuming ~3% store/employee growth
future_rows = []
for company, grp in df.groupby('Company'):
    last = grp[grp.Year == grp.Year.max()].iloc[0]
    future_rows.append({
        'Company': company,
        'Year': 2025,
        'Stores_Worldwide': int(last['Stores_Worldwide'] * 1.03),
        'Employees': int(last['Employees'] * 1.02)
    })
future_df = pd.DataFrame(future_rows)
future_df['Predicted_2025_Sales_Million_USD'] = reg_model.predict(
    future_df[['Year', 'Stores_Worldwide', 'Employees']]
).round(0)
print('🔮 2025 Sales Forecast:')
future_df

## 🤖 Step 6 — Train AI Model #2: Classify the Company (Classification)

We train a **Random Forest Classifier** that identifies *which company* a row of stats belongs to — given `Stores_Worldwide`, `Employees`, `Revenue_Million_USD`, and `Systemwide_Sales_Million_USD`.

In [ ]:
le = LabelEncoder()
df['Company_Label'] = le.fit_transform(df['Company'])

Xc = df[['Stores_Worldwide', 'Employees', 'Revenue_Million_USD', 'Systemwide_Sales_Million_USD']]
yc = df['Company_Label']

Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.3, random_state=42, stratify=yc)

clf_model = RandomForestClassifier(n_estimators=200, random_state=42)
clf_model.fit(Xc_train, yc_train)
yc_pred = clf_model.predict(Xc_test)

print('=' * 60)
print('  AI MODEL #2 — Company Classifier (Random Forest)')
print('=' * 60)
print(f'  Accuracy: {accuracy_score(yc_test, yc_pred) * 100:.1f}%')
print('=' * 60)
print()
print(classification_report(yc_test, yc_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(yc_test, yc_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('🎯  Confusion Matrix — Company Classifier\nM Abu Bakar | SAP: 80429 | Design and Analysis of Algorithms | Sir Karar Haider',
             fontsize=11, fontweight='bold')
add_stamp(ax); plt.tight_layout()
plt.savefig('chart6_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved.')

In [ ]:
# Feature importance from the classifier
importances = pd.Series(clf_model.feature_importances_, index=Xc.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='barh', ax=ax, color='#457b9d')
ax.invert_yaxis()
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('🌲  Feature Importance — What Identifies a Company?\nM Abu Bakar | SAP: 80429 | Design and Analysis of Algorithms | Sir Karar Haider',
             fontsize=11, fontweight='bold')
add_stamp(ax); plt.tight_layout()
plt.savefig('chart7_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance chart saved.')

## 📋 Step 7 — Save Outputs

In [ ]:
df.drop(columns=['Company_Label']).to_csv('fastfood_analysis_output.csv', index=False)
print('✅ Output CSV saved.')

## ✅ Step 8 — Conclusions

| Finding | Detail |
|---|---|
| **Highest Sales (2024)** | **McDonald's** — ~$135B systemwide sales, well ahead of the other two |
| Fastest-Growing Sales | Starbucks — sales nearly doubled from 2015 to 2024 |
| Most Stores | McDonald's — ~43,000 worldwide (2024) |
| Highest Revenue per Store | Starbucks — smaller footprint but strong revenue density |
| Regression Model | Predicts systemwide sales from scale (Year, Stores, Employees) with a good R² fit |
| Classification Model | Random Forest identifies the company from its stats with high accuracy — `Systemwide_Sales_Million_USD` is the most important feature |

**Answer to "jispar zyada sale ho rahi hai":** Based on systemwide sales, **McDonald's** is generating the most sales among the three, followed by Starbucks, then KFC.

---

**Submitted by:** M Abu Bakar  
**SAP ID:** 80429  
**Subject:** Design and Analysis of Algorithms  
**Instructor:** Sir Karar Haider